<a href="https://colab.research.google.com/github/Adamphoenix003/GNN-LinkPrediction/blob/main/SFDDGNN_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SFDDGNN v2 — Fixed Implementation

**Fixes over v1 (which gave ~64% AP):**
- Contrastive loss sign error corrected (was pushing wrong direction)
- X_G recomputed each epoch in Stage 3 (graph_emb is trainable, cannot be cached)
- Dedicated full-graph embedding functions for evaluation (no src/dst leakage)
- Negative sampling uses `force_undirected=True` + filters known edges
- ELSE output cached per-epoch not per-batch (10x faster Stage 1)
- Proper InfoNCE-style contrastive loss replaces broken triplet formula

**Expected results on Cora:** AP ~94-95%, AUC ~93-94%


## Step 1 — Install
> Run this cell then **Runtime → Restart session**

In [1]:
"""
Phase 1 — Structure-Based Learning
Step 1: Graph Embedding Algorithms

Implements the three basis methods used inside ELSE:
  - DeepWalk  (Perozzi et al. 2014)
  - Node2Vec  (Grover & Leskovec 2016)
  - SVD-MF    (truncated SVD of PPMI adjacency matrix)

Each function takes a PyG edge_index and returns a
(num_nodes, embedding_dim) float32 numpy array.
These arrays are later loaded into nn.Embedding for fine-tuning.

Usage:
    from step1_graph_embeddings import DeepWalk, Node2Vec, SVDMF
"""

import numpy as np
import scipy.sparse as sp
from sklearn.utils.extmath import randomized_svd
import torch


# ─────────────────────────────────────────────────────────────
# Shared utility: build adjacency list
# ─────────────────────────────────────────────────────────────

def build_adjacency_list(edge_index: torch.Tensor, num_nodes: int) -> list:
    """
    Convert a PyG COO edge_index into a plain adjacency list.

    Args:
        edge_index : (2, E) long tensor — directed edges (u -> v).
                     For undirected graphs pass to_undirected() first.
        num_nodes  : total number of nodes N.

    Returns:
        adj : list of lists. adj[u] = [v1, v2, ...] neighbors of u.
    """
    adj = [[] for _ in range(num_nodes)]
    for u, v in zip(edge_index[0].tolist(), edge_index[1].tolist()):
        adj[u].append(v)
    return adj


# ─────────────────────────────────────────────────────────────
# Shared utility: Skip-gram with negative sampling
# ─────────────────────────────────────────────────────────────

def skipgram(
    walks:        list,
    num_nodes:    int,
    dim:          int,
    window:       int   = 5,
    num_negative: int   = 5,
    lr:           float = 0.025,
    epochs:       int   = 1,
    seed:         int   = 42,
) -> np.ndarray:
    """
    Skip-gram with negative sampling trained on random walk sequences.

    Intuition
    ---------
    Treat each walk as a sentence, each node as a word.
    For every (center, context) pair within a sliding window:
      - POSITIVE signal: push center and context vectors together.
      - NEGATIVE signal: push center away from `num_negative` randomly
        sampled noise nodes.

    Two matrices are maintained:
      W_in  — "center" vectors  (what we keep as the final embedding)
      W_out — "context" vectors (discarded after training)

    The update rule per (center, context) pair:
      score_pos = sigmoid( W_in[center] · W_out[context] )
      grad_pos  = lr * (1 - score_pos)             # want score -> 1

      for each negative k:
        score_neg = sigmoid( W_in[center] · W_out[k] )
        grad_neg  = -lr * score_neg                 # want score -> 0

      W_out[context] += grad_pos  * W_in[center]
      W_out[k]       += grad_neg  * W_in[center]
      W_in[center]   += grad_pos  * W_out[context]
                      + sum(grad_neg * W_out[k] for k in negatives)

    Negative sampling distribution
    -------------------------------
    Word2Vec samples negatives from unigram^0.75 — this raises the
    probability of common nodes slightly, which prevents rare nodes
    from always being selected as trivially easy negatives.

    Args:
        walks        : list of lists of node ids (walk sequences).
        num_nodes    : vocabulary size (total nodes).
        dim          : embedding dimension.
        window       : context window radius on each side of center.
        num_negative : negative samples per (center, context) pair.
        lr           : learning rate (SGD, no momentum).
        epochs       : passes over all walks.
        seed         : RNG seed for reproducibility.

    Returns:
        W_in : (num_nodes, dim) float32 array — the learned embeddings.
    """
    rng   = np.random.default_rng(seed)
    W_in  = rng.normal(0.0, 0.01, (num_nodes, dim)).astype(np.float32)
    W_out = np.zeros((num_nodes, dim), dtype=np.float32)

    # Unigram^0.75 noise distribution
    counts = np.ones(num_nodes, dtype=np.float64)
    for walk in walks:
        for node in walk:
            counts[node] += 1
    noise_dist  = counts ** 0.75
    noise_dist /= noise_dist.sum()

    def sigmoid(x):
        return 1.0 / (1.0 + np.exp(-np.clip(x, -10.0, 10.0)))

    for epoch in range(epochs):
        rng.shuffle(walks)
        for walk in walks:
            for i, center in enumerate(walk):
                lo = max(0, i - window)
                hi = min(len(walk), i + window + 1)
                context_nodes = [walk[j] for j in range(lo, hi) if j != i]

                for ctx in context_nodes:
                    # Sample negatives
                    negs = rng.choice(num_nodes, size=num_negative,
                                      p=noise_dist)

                    v_c = W_in[center]           # center embedding (d,)

                    # Positive gradient
                    g_pos = lr * (1.0 - sigmoid(v_c @ W_out[ctx]))

                    # Negative gradients  (num_negative,)
                    g_neg = -lr * sigmoid(v_c @ W_out[negs].T)

                    # Update context (output) embeddings
                    W_out[ctx]  += g_pos * v_c
                    W_out[negs] += g_neg[:, np.newaxis] * v_c

                    # Update center (input) embedding
                    W_in[center] += (
                        g_pos * W_out[ctx]
                        + (g_neg[:, np.newaxis] * W_out[negs]).sum(axis=0)
                    )

    return W_in


# ─────────────────────────────────────────────────────────────
# 1. DeepWalk
# ─────────────────────────────────────────────────────────────

def deepwalk_generate_walks(
    adj:         list,
    num_nodes:   int,
    num_walks:   int = 10,
    walk_length: int = 80,
    seed:        int = 42,
) -> list:
    """
    Generate uniform random walk sequences for DeepWalk.

    At every step the walker moves to a uniformly random neighbor.
    If a node has no neighbors it stays in place (isolated node handling).

    Args:
        adj         : adjacency list from build_adjacency_list().
        num_nodes   : N.
        num_walks   : how many walks to start from each node.
        walk_length : number of steps per walk (including start node).
        seed        : RNG seed.

    Returns:
        walks : list of (num_nodes * num_walks) walk sequences.
                Each sequence is a list of node ids of length walk_length.
    """
    rng   = np.random.default_rng(seed)
    nodes = list(range(num_nodes))
    walks = []

    for _ in range(num_walks):
        rng.shuffle(nodes)                   # random order each pass
        for start in nodes:
            walk = [start]
            for _ in range(walk_length - 1):
                cur       = walk[-1]
                neighbors = adj[cur]
                if neighbors:
                    walk.append(int(rng.choice(neighbors)))
                else:
                    walk.append(cur)         # stay if isolated
            walks.append(walk)

    return walks


class DeepWalk:
    """
    DeepWalk (Perozzi et al. 2014).

    Full algorithm:
        1. For every node, generate `num_walks` uniform random walks
           of length `walk_length`.
        2. Train Skip-gram with negative sampling on those walk sequences.
        3. Return the learned (num_nodes, dim) embedding matrix.

    The key difference from Node2Vec: transition probabilities are UNIFORM —
    every neighbor is equally likely regardless of how the walker arrived.

    Example
    -------
        dw = DeepWalk(edge_index, num_nodes=2708, dim=64)
        embeddings = dw.embeddings   # numpy (2708, 64)
    """

    def __init__(
        self,
        edge_index:  torch.Tensor,
        num_nodes:   int,
        dim:         int   = 64,
        num_walks:   int   = 10,
        walk_length: int   = 80,
        window:      int   = 5,
        num_negative:int   = 5,
        sg_epochs:   int   = 1,
        seed:        int   = 42,
        verbose:     bool  = True,
    ):
        self.num_nodes = num_nodes
        self.dim       = dim

        if verbose:
            print(f"[DeepWalk] Building adjacency list...")
        adj = build_adjacency_list(edge_index, num_nodes)

        if verbose:
            total = num_nodes * num_walks
            print(f"[DeepWalk] Generating {total:,} walks "
                  f"({num_walks} per node, length {walk_length})...")
        walks = deepwalk_generate_walks(adj, num_nodes, num_walks,
                                        walk_length, seed)

        if verbose:
            print(f"[DeepWalk] Training Skip-gram on {len(walks):,} walks "
                  f"(window={window}, negatives={num_negative})...")
        self.embeddings = skipgram(
            walks, num_nodes, dim, window, num_negative,
            epochs=sg_epochs, seed=seed,
        )

        if verbose:
            print(f"[DeepWalk] Done. Embedding shape: {self.embeddings.shape}")


# ─────────────────────────────────────────────────────────────
# 2. Node2Vec
# ─────────────────────────────────────────────────────────────

def node2vec_generate_walks(
    adj:         list,
    num_nodes:   int,
    num_walks:   int   = 10,
    walk_length: int   = 80,
    p:           float = 1.0,
    q:           float = 0.5,
    seed:        int   = 42,
) -> list:
    """
    Generate biased random walk sequences for Node2Vec.

    Transition weights from current node `cur` (arrived from `prev`):
    ┌──────────────────────────────────┬────────────┐
    │ Relationship of `next` to `prev` │  Weight    │
    ├──────────────────────────────────┼────────────┤
    │ next == prev  (return)           │  1/p       │
    │ next in neighbors(prev)  (side)  │  1         │
    │ next not in neighbors(prev) (far)│  1/q       │
    └──────────────────────────────────┴────────────┘

    Intuition
    ---------
    p < 1  →  favors returning to previous node (BFS-like, local structure)
    q < 1  →  favors moving to new nodes (DFS-like, global communities)
    p = q = 1  →  reduces exactly to DeepWalk (uniform)

    For the very first step of each walk (no previous node known),
    a neighbor is chosen uniformly.

    Args:
        adj, num_nodes, num_walks, walk_length, seed : same as deepwalk.
        p : return parameter.
        q : in-out parameter.

    Returns:
        walks : same format as deepwalk_generate_walks.
    """
    rng      = np.random.default_rng(seed)
    adj_sets = [set(neighbors) for neighbors in adj]  # O(1) 1-hop lookup
    nodes    = list(range(num_nodes))
    walks    = []

    def transition_probs(prev: int, cur: int):
        """
        Compute normalized transition probabilities from `cur`
        given the walker arrived from `prev`.
        """
        neighbors = adj[cur]
        if not neighbors:
            return neighbors, np.array([])

        weights = np.empty(len(neighbors), dtype=np.float64)
        for i, nxt in enumerate(neighbors):
            if nxt == prev:
                weights[i] = 1.0 / p              # return move
            elif nxt in adj_sets[prev]:
                weights[i] = 1.0                  # distance-1 from prev
            else:
                weights[i] = 1.0 / q              # distance-2+ from prev
        weights /= weights.sum()
        return neighbors, weights

    for _ in range(num_walks):
        rng.shuffle(nodes)
        for start in nodes:
            walk = [start]
            for step in range(walk_length - 1):
                cur       = walk[-1]
                neighbors = adj[cur]

                if not neighbors:
                    walk.append(cur)               # isolated: stay
                    continue

                if step == 0:
                    # First step: no previous node, use uniform
                    walk.append(int(rng.choice(neighbors)))
                else:
                    prev         = walk[-2]
                    nbrs, probs  = transition_probs(prev, cur)
                    walk.append(int(rng.choice(nbrs, p=probs)))

            walks.append(walk)

    return walks


class Node2Vec:
    """
    Node2Vec (Grover & Leskovec 2016).

    Same pipeline as DeepWalk but uses biased walks controlled by p and q.

    p=1, q=1  →  identical to DeepWalk
    p=1, q<1  →  DFS bias: captures community / cluster membership
    p<1, q=1  →  BFS bias: captures local structural roles

    The paper found q=0.5 generally works well for link prediction
    (slightly DFS-biased to capture community overlap).

    Example
    -------
        n2v = Node2Vec(edge_index, num_nodes=2708, dim=64, p=1.0, q=0.5)
        embeddings = n2v.embeddings   # numpy (2708, 64)
    """

    def __init__(
        self,
        edge_index:   torch.Tensor,
        num_nodes:    int,
        dim:          int   = 64,
        num_walks:    int   = 10,
        walk_length:  int   = 80,
        p:            float = 1.0,
        q:            float = 0.5,
        window:       int   = 5,
        num_negative: int   = 5,
        sg_epochs:    int   = 1,
        seed:         int   = 42,
        verbose:      bool  = True,
    ):
        self.num_nodes = num_nodes
        self.dim       = dim

        if verbose:
            print(f"[Node2Vec] Building adjacency list...")
        adj = build_adjacency_list(edge_index, num_nodes)

        if verbose:
            total = num_nodes * num_walks
            print(f"[Node2Vec] Generating {total:,} biased walks "
                  f"(p={p}, q={q}, length={walk_length})...")
        walks = node2vec_generate_walks(adj, num_nodes, num_walks,
                                        walk_length, p, q, seed)

        if verbose:
            print(f"[Node2Vec] Training Skip-gram on {len(walks):,} walks...")
        self.embeddings = skipgram(
            walks, num_nodes, dim, window, num_negative,
            epochs=sg_epochs, seed=seed,
        )

        if verbose:
            print(f"[Node2Vec] Done. Embedding shape: {self.embeddings.shape}")


# ─────────────────────────────────────────────────────────────
# 3. SVD Matrix Factorisation
# ─────────────────────────────────────────────────────────────

def build_ppmi_matrix(
    edge_index: torch.Tensor,
    num_nodes:  int,
) -> sp.csr_matrix:
    """
    Build the PPMI (Positive Pointwise Mutual Information) matrix
    from a graph's edge_index.

    Steps
    -----
    1. Build symmetric binary adjacency matrix A (N x N).
    2. Compute PMI for each existing edge (i, j):

           PMI(i,j) = log( A[i,j] * total / (row_sum[i] * col_sum[j]) )

       Intuition: how much MORE often do i and j co-occur (are connected)
       compared to what we'd expect if connections were random?
       - High PMI = connection is surprising given degrees → meaningful
       - Low / negative PMI = connection expected by chance → downweighted

    3. PPMI = max(0, PMI)  — clip negatives to zero.

    Why PPMI instead of raw adjacency?
    -----------------------------------
    Raw A[i,j] = 1 for all edges regardless of degree.
    A hub node (degree 1000) connected to node j is EXPECTED —
    there's nothing informative about that edge.
    A rare low-degree node connected to another rare node is SURPRISING —
    that connection carries real information.
    PPMI captures this by re-weighting each edge by its "surprise".

    Returns:
        ppmi : (N, N) sparse CSR matrix of PPMI values.
    """
    src  = edge_index[0].numpy()
    dst  = edge_index[1].numpy()
    data = np.ones(len(src), dtype=np.float32)

    A = sp.csr_matrix((data, (src, dst)), shape=(num_nodes, num_nodes))
    A = (A + A.T)                         # make symmetric (undirected)
    A.data = np.clip(A.data, 0.0, 1.0)   # binarise double-counted edges

    total   = float(A.sum())
    row_sum = np.asarray(A.sum(axis=1)).flatten()   # degree of each node
    col_sum = np.asarray(A.sum(axis=0)).flatten()   # (same for undirected)

    coo = A.tocoo()
    rows, cols, vals = coo.row, coo.col, coo.data.copy()

    # PMI = log( p(i,j) / (p(i)*p(j)) )
    #     = log( A[i,j] * total / (row_sum[i] * col_sum[j]) )
    denom = row_sum[rows] * col_sum[cols]
    safe  = denom > 0
    pmi   = np.zeros_like(vals, dtype=np.float32)
    pmi[safe] = np.log(
        vals[safe] * total / denom[safe] + 1e-8
    ).astype(np.float32)

    ppmi_vals = np.maximum(pmi, 0.0)     # PPMI: clip negatives

    return sp.csr_matrix(
        (ppmi_vals, (rows, cols)),
        shape=(num_nodes, num_nodes),
        dtype=np.float32,
    )


class SVDMF:
    """
    SVD-based Matrix Factorisation (Levy & Goldberg 2014 applied to graphs).

    Full algorithm
    --------------
    1. Build PPMI matrix from the graph (or raw adjacency if use_ppmi=False).
    2. Run truncated SVD:   PPMI ≈ U @ diag(S) @ Vt
       where U is (N, k), S is (k,), Vt is (k, N).
    3. Node embedding = U @ diag( sqrt(S) )   shape: (N, k)

    Why sqrt(S) and not S?
    ----------------------
    We want:  embedding_i · embedding_j  ≈  PPMI[i, j]

    If embedding = U @ diag(sqrt(S)), then:
        embedding @ embedding.T
        = U @ diag(sqrt(S)) @ diag(sqrt(S)) @ U.T
        = U @ diag(S) @ U.T
        ≈ PPMI   (by the SVD approximation)

    Using the full S in one side would give U @ S @ U.T, which is
    the "asymmetric" version and doesn't satisfy the dot-product property.

    Why truncated SVD?
    ------------------
    The full SVD of an N×N matrix is O(N³) — infeasible for large graphs.
    Truncated / randomized SVD only computes the top-k singular vectors,
    which is O(N·k) and much faster. We only need k=embedding_dim vectors.

    Example
    -------
        svd = SVDMF(edge_index, num_nodes=2708, dim=64)
        embeddings = svd.embeddings   # numpy (2708, 64)
        # svd.singular_values shows the top-k singular values
    """

    def __init__(
        self,
        edge_index: torch.Tensor,
        num_nodes:  int,
        dim:        int  = 64,
        use_ppmi:   bool = True,
        n_iter:     int  = 5,
        seed:       int  = 42,
        verbose:    bool = True,
    ):
        self.num_nodes = num_nodes
        self.dim       = dim

        if verbose:
            mat_type = "PPMI" if use_ppmi else "adjacency"
            print(f"[SVD MF] Building {mat_type} matrix "
                  f"({num_nodes} x {num_nodes})...")

        if use_ppmi:
            matrix = build_ppmi_matrix(edge_index, num_nodes)
        else:
            # Raw binary adjacency (symmetric)
            src  = edge_index[0].numpy()
            dst  = edge_index[1].numpy()
            A    = sp.csr_matrix(
                (np.ones(len(src), np.float32), (src, dst)),
                shape=(num_nodes, num_nodes))
            matrix = (A + A.T)
            matrix.data = np.clip(matrix.data, 0, 1)

        if verbose:
            nnz = matrix.nnz
            print(f"[SVD MF] Matrix sparsity: {nnz:,} non-zeros "
                  f"({100*nnz/num_nodes**2:.3f}% dense). "
                  f"Running truncated SVD (k={dim})...")

        # Randomized SVD — much faster than full SVD for sparse matrices
        U, S, _Vt = randomized_svd(
            matrix,
            n_components=dim,
            n_iter=n_iter,
            random_state=seed,
        )

        # Absorb sqrt of singular values into U: embedding = U * sqrt(S)
        # Shape: (N, dim)
        self.embeddings      = (U * np.sqrt(S)[np.newaxis, :]).astype(np.float32)
        self.singular_values = S

        if verbose:
            print(f"[SVD MF] Done. Embedding shape: {self.embeddings.shape}")
            print(f"[SVD MF] Top-5 singular values: "
                  f"{np.round(S[:5], 3)}")


# ─────────────────────────────────────────────────────────────
# Quick smoke test
# ─────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("=" * 50)
    print("Smoke test with a tiny 6-node ring graph")
    print("=" * 50)

    # 0-1-2-3-4-5-0  (ring)
    src = torch.tensor([0,1,2,3,4,5, 1,2,3,4,5,0])
    dst = torch.tensor([1,2,3,4,5,0, 0,1,2,3,4,5])
    ei  = torch.stack([src, dst])
    N   = 6
    D   = 4   # tiny dim for speed

    print("\n--- DeepWalk ---")
    dw = DeepWalk(ei, N, dim=D, num_walks=3, walk_length=10, sg_epochs=1)
    print(f"  embeddings[0]: {dw.embeddings[0].round(3)}")

    print("\n--- Node2Vec (p=1, q=0.5) ---")
    n2v = Node2Vec(ei, N, dim=D, num_walks=3, walk_length=10,
                   p=1.0, q=0.5, sg_epochs=1)
    print(f"  embeddings[0]: {n2v.embeddings[0].round(3)}")

    print("\n--- SVD MF (PPMI) ---")
    svd = SVDMF(ei, N, dim=D, use_ppmi=True)
    print(f"  embeddings[0]: {svd.embeddings[0].round(3)}")

    print("\nAll three algorithms ran successfully.")

Smoke test with a tiny 6-node ring graph

--- DeepWalk ---
[DeepWalk] Building adjacency list...
[DeepWalk] Generating 18 walks (3 per node, length 10)...
[DeepWalk] Training Skip-gram on 18 walks (window=5, negatives=5)...
[DeepWalk] Done. Embedding shape: (6, 4)
  embeddings[0]: [-0.619 -1.219  0.966  0.242]

--- Node2Vec (p=1, q=0.5) ---
[Node2Vec] Building adjacency list...
[Node2Vec] Generating 18 biased walks (p=1.0, q=0.5, length=10)...
[Node2Vec] Training Skip-gram on 18 walks...
[Node2Vec] Done. Embedding shape: (6, 4)
  embeddings[0]: [-0.4   -1.057  1.007  0.174]

--- SVD MF (PPMI) ---
[SVD MF] Building PPMI matrix (6 x 6)...
[SVD MF] Matrix sparsity: 12 non-zeros (33.333% dense). Running truncated SVD (k=4)...
[SVD MF] Done. Embedding shape: (6, 4)
[SVD MF] Top-5 singular values: [2.197 2.197 1.099 1.099]
  embeddings[0]: [0.856 0.    0.771 0.006]

All three algorithms ran successfully.


## Step 2 — Imports

In [6]:
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.0 MB/s eta 0:00:00


In [7]:
"""
Phase 1 — Structure-Based Learning
Step 2: Ensemble Learning Structure Encoder (ELSE)

Takes the three pre-computed embedding arrays from Step 1 and builds
the ELSE module described in Section 3.1.1 / Equations 1-2 of the paper.

ELSE does two things:
  1. Wraps each embedding array in a trainable nn.Embedding so the
     pre-computed vectors can be fine-tuned by backprop.
  2. Concatenates all three and fuses them through a 2-layer MLP
     to produce the initial structural node embedding X^B.

Usage:
    from step1_graph_embeddings import DeepWalk, Node2Vec, SVDMF
    from step2_else_encoder import ELSEEncoder

    dw  = DeepWalk(train_edge_index, num_nodes, dim=64)
    n2v = Node2Vec(train_edge_index, num_nodes, dim=64, p=1.0, q=0.5)
    svd = SVDMF(train_edge_index, num_nodes, dim=64)

    else_enc = ELSEEncoder(dw, n2v, svd, output_dim=256)
    X_B = else_enc(node_ids)   # (N, 256)
"""

import torch
import torch.nn as nn
import numpy as np




# ─────────────────────────────────────────────────────────────
# Helper: wrap a numpy array in a trainable nn.Embedding
# ─────────────────────────────────────────────────────────────

def array_to_embedding(
    array:     np.ndarray,
    trainable: bool = True,
) -> nn.Embedding:
    """
    Load a (N, d) numpy array into an nn.Embedding.

    Setting requires_grad=True means the pre-computed weights
    will continue to be updated by backprop during SFDDGNN training —
    the graph algorithms provide a good starting point, backprop refines it.

    Setting requires_grad=False freezes the weights (useful for ablation).

    Args:
        array     : (N, d) float32 numpy array from DeepWalk / Node2Vec / SVD.
        trainable : if True, weights are updated during training.

    Returns:
        emb : nn.Embedding with weight initialised from `array`.
    """
    N, d = array.shape
    emb  = nn.Embedding(N, d)
    emb.weight = nn.Parameter(
        torch.from_numpy(array),
        requires_grad=trainable,
    )
    return emb


# ─────────────────────────────────────────────────────────────
# ELSE Encoder
# ─────────────────────────────────────────────────────────────

class ELSEEncoder(nn.Module):
    """
    Ensemble Learning Structure Encoder (ELSE) — Section 3.1.1, Eq 1-2.

    What it does
    ------------
    Takes three pre-computed embedding arrays (DeepWalk, Node2Vec, SVD-MF),
    loads them into trainable nn.Embedding tables, then fuses them with a
    2-layer MLP to produce the initial structural embedding X^B.

    Paper Equation 2:
        X^B = MLP( DeepWalk(ids) || Node2Vec(ids) || SVDMF(ids) )

    where || is concatenation along the feature dimension.

    Why concatenate instead of average or add?
    -------------------------------------------
    Each method captures a different aspect of topology:
      - DeepWalk captures uniform random walk statistics (global reachability)
      - Node2Vec captures biased walk statistics (community structure)
      - SVD-MF captures spectral / co-occurrence structure (PPMI-weighted)

    Averaging would discard the complementary information.
    Concatenation preserves all three signals; the MLP learns how to
    weight and combine them based on what's useful for link prediction.

    Architecture:
        [dw_emb || n2v_emb || svd_emb]  →  Linear(3d, h) → ReLU → Linear(h, output_dim)
        shape: (N, 3*basis_dim) → (N, output_dim)

    Args:
        deepwalk    : DeepWalk instance (has .embeddings numpy array).
        node2vec    : Node2Vec instance (has .embeddings numpy array).
        svd_mf      : SVDMF instance (has .embeddings numpy array).
        output_dim  : output dimension of the MLP (fed into GraphSAGE).
                      Paper default: 256.
        trainable   : whether to fine-tune the pre-computed embedding weights.
    """

    def __init__(
        self,
        deepwalk:   DeepWalk,
        node2vec:   Node2Vec,
        svd_mf:     SVDMF,
        output_dim: int  = 256,
        trainable:  bool = True,
    ):
        super().__init__()

        # Validate all three have the same basis dimension
        d_dw  = deepwalk.embeddings.shape[1]
        d_n2v = node2vec.embeddings.shape[1]
        d_svd = svd_mf.embeddings.shape[1]
        assert d_dw == d_n2v == d_svd, (
            f"Basis dims must match: DeepWalk={d_dw}, "
            f"Node2Vec={d_n2v}, SVD={d_svd}"
        )
        self.basis_dim  = d_dw
        self.output_dim = output_dim

        # Wrap each array in a trainable nn.Embedding  (Eq. 1)
        self.dw_emb  = array_to_embedding(deepwalk.embeddings, trainable)
        self.n2v_emb = array_to_embedding(node2vec.embeddings, trainable)
        self.svd_emb = array_to_embedding(svd_mf.embeddings,  trainable)

        # 2-layer MLP for fusion  (Eq. 2)
        self.mlp = nn.Sequential(
            nn.Linear(self.basis_dim * 3, output_dim),
            nn.ReLU(),
            nn.Linear(output_dim, output_dim),
        )

        self._init_mlp()

        print(f"[ELSE] Encoder ready.")
        print(f"  Basis dim   : {self.basis_dim} (x3 = {self.basis_dim*3})")
        print(f"  Output dim  : {output_dim}")
        print(f"  Trainable   : {trainable}")
        print(f"  Total params: {sum(p.numel() for p in self.parameters()):,}")

    def _init_mlp(self):
        """Xavier uniform initialisation for MLP weights."""
        for layer in self.mlp:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                nn.init.zeros_(layer.bias)

    def forward(self, node_ids: torch.Tensor) -> torch.Tensor:
        """
        Produce initial structural embedding X^B for a set of nodes.

        Args:
            node_ids : (N,) or (B,) long tensor of node indices.

        Returns:
            X_B : (..., output_dim) float tensor — initial structural embeddings.

        Step-by-step:
            1. Look up each embedding table at the given indices.
               Each returns (..., basis_dim).
            2. Concatenate along the last dimension → (..., 3 * basis_dim).
            3. Pass through the 2-layer MLP → (..., output_dim).
        """
        h_dw  = self.dw_emb(node_ids)    # (..., basis_dim)
        h_n2v = self.n2v_emb(node_ids)   # (..., basis_dim)
        h_svd = self.svd_emb(node_ids)   # (..., basis_dim)

        # Concatenate  (Eq. 2)
        h_cat = torch.cat([h_dw, h_n2v, h_svd], dim=-1)  # (..., 3*basis_dim)

        # Fuse through MLP
        return self.mlp(h_cat)            # (..., output_dim)  = X^B


# ─────────────────────────────────────────────────────────────
# Convenience factory: build ELSE from scratch in one call
# ─────────────────────────────────────────────────────────────

def build_else_encoder(
    edge_index:  torch.Tensor,
    num_nodes:   int,
    basis_dim:   int   = 64,
    output_dim:  int   = 256,
    num_walks:   int   = 10,
    walk_length: int   = 80,
    n2v_p:       float = 1.0,
    n2v_q:       float = 0.5,
    use_ppmi:    bool  = True,
    trainable:   bool  = True,
    verbose:     bool  = True,
) -> ELSEEncoder:
    """
    Run all three graph embedding algorithms and build the ELSE encoder.

    This is the entry point for Phase 1 Step 2.

    Args:
        edge_index  : (2, E) training graph edges (undirected).
        num_nodes   : total nodes N.
        basis_dim   : embedding dimension for each of the 3 basis methods.
        output_dim  : MLP output dimension (fed into GraphSAGE next).
        num_walks   : walks per node for DeepWalk / Node2Vec.
        walk_length : steps per walk.
        n2v_p       : Node2Vec return parameter.
        n2v_q       : Node2Vec in-out parameter.
        use_ppmi    : use PPMI normalisation for SVD-MF.
        trainable   : allow fine-tuning of embedding weights.
        verbose     : print progress.

    Returns:
        else_enc : ELSEEncoder module ready for forward passes.
    """
    if verbose:
        print("=" * 55)
        print("Building ELSE Encoder")
        print("=" * 55)

    dw  = DeepWalk(edge_index, num_nodes, dim=basis_dim,
                   num_walks=num_walks, walk_length=walk_length,
                   verbose=verbose)

    n2v = Node2Vec(edge_index, num_nodes, dim=basis_dim,
                   num_walks=num_walks, walk_length=walk_length,
                   p=n2v_p, q=n2v_q, verbose=verbose)

    svd = SVDMF(edge_index, num_nodes, dim=basis_dim,
                use_ppmi=use_ppmi, verbose=verbose)

    else_enc = ELSEEncoder(dw, n2v, svd,
                            output_dim=output_dim,
                            trainable=trainable)
    return else_enc


# ─────────────────────────────────────────────────────────────
# Smoke test
# ─────────────────────────────────────────────────────────────

if __name__ == "__main__":
    from torch_geometric.utils import to_undirected

    print("=" * 55)
    print("Smoke test: ELSE on a 10-node graph")
    print("=" * 55)

    # Small random graph
    src = torch.tensor([0,1,2,3,4,5,6,7,8,9,0,2,4,6,8])
    dst = torch.tensor([1,2,3,4,5,6,7,8,9,0,5,7,9,1,3])
    ei  = to_undirected(torch.stack([src, dst]))
    N   = 10

    else_enc = build_else_encoder(
        edge_index  = ei,
        num_nodes   = N,
        basis_dim   = 8,
        output_dim  = 16,
        num_walks   = 3,
        walk_length = 10,
    )

    node_ids = torch.arange(N)
    X_B      = else_enc(node_ids)

    print(f"\nForward pass output X_B shape : {X_B.shape}")
    print(f"Expected                      : ({N}, 16)")
    print(f"X_B[0]: {X_B[0].detach().numpy().round(3)}")

    # Verify gradients flow
    loss = X_B.sum()
    loss.backward()
    grads_ok = all(
        p.grad is not None
        for p in else_enc.parameters()
        if p.requires_grad
    )
    print(f"\nGradients flow to all params  : {grads_ok}")
    print("\nELSE encoder test passed.")

Smoke test: ELSE on a 10-node graph
Building ELSE Encoder
[DeepWalk] Building adjacency list...
[DeepWalk] Generating 30 walks (3 per node, length 10)...
[DeepWalk] Training Skip-gram on 30 walks (window=5, negatives=5)...
[DeepWalk] Done. Embedding shape: (10, 8)
[Node2Vec] Building adjacency list...
[Node2Vec] Generating 30 biased walks (p=1.0, q=0.5, length=10)...
[Node2Vec] Training Skip-gram on 30 walks...
[Node2Vec] Done. Embedding shape: (10, 8)
[SVD MF] Building PPMI matrix (10 x 10)...
[SVD MF] Matrix sparsity: 30 non-zeros (30.000% dense). Running truncated SVD (k=8)...
[SVD MF] Done. Embedding shape: (10, 8)
[SVD MF] Top-5 singular values: [3.612 3.612 1.948 1.948 1.948]
[ELSE] Encoder ready.
  Basis dim   : 8 (x3 = 24)
  Output dim  : 16
  Trainable   : True
  Total params: 912

Forward pass output X_B shape : torch.Size([10, 16])
Expected                      : (10, 16)
X_B[0]: [ 0.162 -0.491  0.193 -0.57  -0.351 -0.479  0.632  0.694 -0.007  0.192
 -0.454  0.449  0.259  0.

In [8]:
"""
Phase 1 — Structure-Based Learning
Step 3: GraphSAGE-Based Structure Encoder

Implements the modified GraphSAGE described in Section 3.1.2 / Equations 3-6.

Two key modifications over vanilla GraphSAGE:
  1. Edge embeddings are updated after each message-passing layer
     using residual connections (Eq. 4). This gives the model a
     direct edge-level representation, not just node-level.
  2. Training uses an edge-based binary cross-entropy loss (Eq. 5)
     so the encoder is optimized directly for link prediction,
     not as a side-effect of node classification.

Public interface:
    get_node_embeddings(x, edge_index)
        → H_S: (N, out_dim)   — used by Stages 2 and 3 of SFDDGNN
    forward_with_edges(x, edge_index, src, dst)
        → H_S, logits         — used during Stage 1 training only
    structure_loss(logits, labels)
        → scalar BCE loss     — Eq. 5

Usage:
    struct_enc = GraphSAGEStructureEncoder(in_dim=256, out_dim=128)
    H_S, logits = struct_enc.forward_with_edges(X_B, train_ei, src, dst)
    loss = GraphSAGEStructureEncoder.structure_loss(logits, labels)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv


class GraphSAGEStructureEncoder(nn.Module):
    """
    Modified GraphSAGE for link prediction (Section 3.1.2, Eq 3-6).

    Architecture
    ------------
    Input: X^B from ELSE encoder, shape (N, in_dim)

    Node path (2-layer GraphSAGE):
        h^1_i = ReLU( W_N · Aggregator( h^0_i ∪ {h^0_u : u ∈ N(i)} ) )
        h^2_i = W_N · Aggregator( h^1_i ∪ {h^1_u : u ∈ N(i)} )
        H_S   = h^2   shape (N, out_dim)

    Edge path (initialized from node embeddings, updated residually):
        h^0_{ij} = ReLU( W_E · [ x_i || x_j ] )         # Eq. 3
        h^1_{ij} = ReLU( W_E · [ h^0_{ij} || h^2_i || h^2_j ] )  # Eq. 4
        logit_ij = W_R · h^1_{ij}                        # ReadOut

    Loss:
        L_S = BCE( sigmoid(logit_ij), y_ij )              # Eq. 5

    Why separate edge embeddings?
    -----------------------------
    Standard GraphSAGE only produces node embeddings. To predict whether
    an edge (i,j) exists, you then compute something like sigmoid(h_i · h_j).
    This is a weak signal for edges where nodes have similar representations
    for unrelated reasons.

    Maintaining an explicit edge embedding h_{ij} lets the model track
    the pairwise relationship directly. The residual update in Eq. 4
    combines: what the edge looked like at layer 0 + what the endpoint
    nodes look like after message passing — giving the edge embedding
    both early and late structural signals.

    Why `get_node_embeddings` is separate from `forward_with_edges`
    ---------------------------------------------------------------
    `forward_with_edges` needs src/dst to compute the edge embedding path.
    For evaluation and Stages 2/3 we only need H_S (the node embeddings),
    and we don't want to pass val/test src/dst through the edge update layers
    (that would be data leakage). `get_node_embeddings` runs only the
    GraphSAGE conv layers on the full graph.

    Args:
        in_dim     : input feature dimension (= ELSE output_dim = 256).
        hidden_dim : hidden dimension inside GraphSAGE (paper default: 256).
        out_dim    : output dimension of H_S (paper default: 128).
        dropout    : dropout rate applied after layer 1 (paper default: 0.5).
    """

    def __init__(
        self,
        in_dim:     int   = 256,
        hidden_dim: int   = 256,
        out_dim:    int   = 128,
        dropout:    float = 0.5,
    ):
        super().__init__()

        # GraphSAGE message-passing layers (Eq. 3 — node updates)
        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, out_dim)
        self.drop  = nn.Dropout(dropout)

        # Edge embedding update layers (Eq. 3-4)
        # Layer 0 edge update: takes concatenated endpoint features
        self.edge_lin0 = nn.Linear(in_dim * 2, hidden_dim)
        # Layer 1 edge update: takes [h_edge0 || h_i_final || h_j_final]
        self.edge_lin1 = nn.Linear(hidden_dim + out_dim * 2, out_dim)

        # ReadOut: edge embedding → link probability score (Eq. 5)
        self.readout   = nn.Linear(out_dim, 1)

        self._init_weights()

    def _init_weights(self):
        for m in [self.edge_lin0, self.edge_lin1, self.readout]:
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)

    # ── Public method 1: full-graph node embeddings only ─────────────────
    def get_node_embeddings(
        self,
        x:          torch.Tensor,
        edge_index: torch.Tensor,
    ) -> torch.Tensor:
        """
        Run the two GraphSAGE layers on the full graph to produce H_S.

        Used by: evaluation, Stage 2, Stage 3.
        Does NOT use src/dst — no data leakage from val/test pairs.

        Args:
            x          : (N, in_dim) initial node features from ELSE.
            edge_index : (2, E) training graph.

        Returns:
            H_S : (N, out_dim) structure-only node embeddings.
        """
        # Layer 1: aggregate neighbors, activate
        h1 = F.relu(self.conv1(x, edge_index))   # (N, hidden_dim)
        h1 = self.drop(h1)

        # Layer 2: aggregate again, no activation (pre-fusion flexibility)
        h2 = self.conv2(h1, edge_index)           # (N, out_dim)

        return h2   # this is H_S

    # ── Public method 2: node embeddings + edge scores ────────────────────
    def forward_with_edges(
        self,
        x:          torch.Tensor,
        edge_index: torch.Tensor,
        src:        torch.Tensor,
        dst:        torch.Tensor,
    ) -> tuple:
        """
        Full forward pass including the edge embedding path.

        Used by: Stage 1 training ONLY.

        Step-by-step:
            1. Initial edge embedding h^0_{ij} from raw ELSE features (Eq. 3).
               This captures the raw feature relationship before any message passing.
            2. Run 2-layer GraphSAGE on ALL nodes to get H_S (Eq. 3).
            3. Residual edge update h^1_{ij} from
               [h^0_{ij} || H_S[src] || H_S[dst]]   (Eq. 4).
               This blends the early edge signal with the post-message-passing
               node representations.
            4. ReadOut: linear layer on h^1_{ij} → link logit (Eq. 5).

        Args:
            x          : (N, in_dim) ELSE output.
            edge_index : (2, E) training graph.
            src        : (B,) source node indices of target pairs.
            dst        : (B,) destination node indices of target pairs.

        Returns:
            H_S    : (N, out_dim) node embeddings — structure-only.
            logits : (B,) raw link prediction scores (before sigmoid).
        """
        # ── Step 1: Initial edge embedding (Eq. 3) ──────────────────────
        # Concatenate endpoint features from X^B (raw, before message passing)
        # h^0_{ij} = ReLU( W_E · [x_i || x_j] )
        h_edge0 = F.relu(
            self.edge_lin0(
                torch.cat([x[src], x[dst]], dim=-1)
            )
        )   # (B, hidden_dim)

        # ── Step 2: 2-layer GraphSAGE (Eq. 3 — node updates) ────────────
        h2 = self.get_node_embeddings(x, edge_index)   # (N, out_dim)  = H_S

        # ── Step 3: Residual edge update (Eq. 4) ────────────────────────
        # h^1_{ij} = ReLU( W_E · [h^0_{ij} || h^2_i || h^2_j] )
        # Combines:
        #   h_edge0  — early structural signal (before message passing)
        #   h2[src]  — endpoint i after full message passing
        #   h2[dst]  — endpoint j after full message passing
        h_edge1 = F.relu(
            self.edge_lin1(
                torch.cat([h_edge0, h2[src], h2[dst]], dim=-1)
            )
        )   # (B, out_dim)

        # ── Step 4: ReadOut — edge embedding → link logit (Eq. 5) ───────
        logits = self.readout(h_edge1).squeeze(-1)   # (B,)

        return h2, logits

    # ── Loss function ─────────────────────────────────────────────────────
    @staticmethod
    def structure_loss(
        logits: torch.Tensor,
        labels: torch.Tensor,
    ) -> torch.Tensor:
        """
        Edge-based binary cross-entropy loss (Eq. 5).

        L_S = - 1/|B| * sum( y*log(p) + (1-y)*log(1-p) )

        where p = sigmoid(logit) and y ∈ {0, 1}.

        Labels:
            1 = positive edge (exists in training graph)
            0 = negative edge (sampled non-edge)

        Args:
            logits : (B,) raw scores from readout (before sigmoid).
            labels : (B,) float tensor of 0/1 labels.

        Returns:
            loss : scalar tensor.
        """
        return F.binary_cross_entropy_with_logits(logits, labels.float())


# ─────────────────────────────────────────────────────────────
# Smoke test
# ─────────────────────────────────────────────────────────────

if __name__ == "__main__":
    from torch_geometric.utils import to_undirected

    print("=" * 55)
    print("Smoke test: GraphSAGE Structure Encoder")
    print("=" * 55)

    N = 10
    D_IN  = 16   # pretend ELSE output_dim = 16 (small for speed)
    D_OUT = 8

    # Small ring graph
    src = torch.tensor([0,1,2,3,4,5,6,7,8,9])
    dst = torch.tensor([1,2,3,4,5,6,7,8,9,0])
    ei  = to_undirected(torch.stack([src, dst]))

    # Fake X^B from ELSE
    x = torch.randn(N, D_IN)

    # Target node pairs (mix of pos and neg)
    pair_src = torch.tensor([0, 1, 2, 5])
    pair_dst = torch.tensor([1, 3, 3, 7])
    labels   = torch.tensor([1, 0, 0, 1], dtype=torch.float)

    enc = GraphSAGEStructureEncoder(in_dim=D_IN, hidden_dim=16, out_dim=D_OUT)

    # Test get_node_embeddings (no src/dst needed)
    H_S = enc.get_node_embeddings(x, ei)
    print(f"\nget_node_embeddings:")
    print(f"  H_S shape : {H_S.shape}   expected: ({N}, {D_OUT})")

    # Test forward_with_edges (Stage 1 training)
    H_S2, logits = enc.forward_with_edges(x, ei, pair_src, pair_dst)
    print(f"\nforward_with_edges:")
    print(f"  H_S shape    : {H_S2.shape}      expected: ({N}, {D_OUT})")
    print(f"  logits shape : {logits.shape}  expected: ({len(pair_src)},)")
    print(f"  logits       : {logits.detach().numpy().round(3)}")

    # Test loss
    loss = GraphSAGEStructureEncoder.structure_loss(logits, labels)
    print(f"\nstructure_loss: {loss.item():.4f}")

    # Test backward
    loss.backward()
    grads_ok = all(p.grad is not None for p in enc.parameters())
    print(f"Gradients flow : {grads_ok}")
    print("\nGraphSAGE encoder test passed.")

Smoke test: GraphSAGE Structure Encoder

get_node_embeddings:
  H_S shape : torch.Size([10, 8])   expected: (10, 8)

forward_with_edges:
  H_S shape    : torch.Size([10, 8])      expected: (10, 8)
  logits shape : torch.Size([4])  expected: (4,)
  logits       : [1.518 0.084 0.798 0.651]

structure_loss: 0.6309
Gradients flow : True

GraphSAGE encoder test passed.


In [9]:
"""
Phase 1 — Structure-Based Learning
Step 4: Data Loading and Edge Splitting

Handles:
  - Loading Cora / PubMed / CiteSeer from PyG
  - 80/10/10 train/val/test edge split
  - Negative edge sampling (force_undirected, exclude known edges)
  - Returning everything as plain tensors ready for the training loop

Usage:
    from step4_data import load_and_split

    d = load_and_split('Cora')

    # Training graph
    d['train_edge_index']   # (2, E_train) — undirected training graph
    d['train_pos_src']      # (E_train/2,) — positive edge sources
    d['train_pos_dst']      # (E_train/2,) — positive edge destinations
    d['train_neg_src']      # (E_train/2,) — negative edge sources
    d['train_neg_dst']      # (E_train/2,) — negative edge destinations

    # Val / test (same structure)
    d['val_pos_src'],  d['val_pos_dst'],  d['val_neg_src'],  d['val_neg_dst']
    d['test_pos_src'], d['test_pos_dst'], d['test_neg_src'], d['test_neg_dst']

    # Node info
    d['node_ids']       # (N,) arange tensor
    d['node_features']  # (N, F) float tensor
    d['num_nodes']      # int
    d['feat_dim']       # int
"""

import torch
import numpy as np
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import to_undirected, negative_sampling
import torch_geometric.transforms as T


def load_and_split(
    dataset_name: str  = 'Cora',
    data_root:    str  = './data',
    train_ratio:  float = 0.80,
    val_ratio:    float = 0.10,
    seed:         int   = 42,
    verbose:      bool  = True,
) -> dict:
    """
    Load a citation dataset and split edges into train / val / test.

    Split logic
    -----------
    Only undirected edges are used (we keep src < dst to avoid counting
    each edge twice). They are shuffled and split:
        train : first 80%
        val   : next  10%
        test  : last  10%

    The training graph is the undirected version of the train positive edges.
    Val and test edges are NOT added to the training graph — they are the
    held-out set the model must predict.

    Negative sampling
    -----------------
    For each split we sample an equal number of non-edges:
      - force_undirected=True ensures symmetric pairs (no (u,v) without (v,u))
      - The full edge_index is passed as the exclusion set so sampled
        negatives never overlap with any known edge (train/val/test)

    Args:
        dataset_name : 'Cora', 'PubMed', or 'CiteSeer'.
        data_root    : directory to cache downloaded datasets.
        train_ratio  : fraction of edges for training (default 0.80).
        val_ratio    : fraction of edges for validation (default 0.10).
        seed         : random seed for reproducibility.
        verbose      : print dataset statistics.

    Returns:
        dict with keys documented in the module docstring above.
    """
    torch.manual_seed(seed)
    np.random.seed(seed)

    # ── Load dataset ──────────────────────────────────────────────────────
    dataset = Planetoid(
        root      = data_root,
        name      = dataset_name,
        transform = T.NormalizeFeatures(),
    )
    data = dataset[0]

    # Make undirected (some datasets have directed edges)
    data.edge_index = to_undirected(data.edge_index)

    num_nodes = data.num_nodes
    feat_dim  = data.x.shape[1]

    if verbose:
        num_edges = data.edge_index.shape[1] // 2
        print(f"Dataset       : {dataset_name}")
        print(f"Nodes         : {num_nodes}")
        print(f"Edges         : {num_edges}")
        print(f"Feature dim   : {feat_dim}")

    # ── Edge split ────────────────────────────────────────────────────────
    # Keep only one direction (src < dst) to avoid double-counting
    mask  = data.edge_index[0] < data.edge_index[1]
    edges = data.edge_index[:, mask]                    # (2, E_unique)

    # Shuffle
    perm  = torch.randperm(edges.shape[1])
    edges = edges[:, perm]

    n       = edges.shape[1]
    n_train = int(train_ratio * n)
    n_val   = int(val_ratio   * n)
    n_test  = n - n_train - n_val

    train_pos = edges[:, :n_train]
    val_pos   = edges[:, n_train : n_train + n_val]
    test_pos  = edges[:, n_train + n_val:]

    # Training graph: undirected version of train positive edges only
    # Val and test edges are EXCLUDED (they are the prediction targets)
    train_edge_index = to_undirected(train_pos)

    if verbose:
        print(f"\nEdge split (80/10/10):")
        print(f"  Train pos : {n_train}")
        print(f"  Val   pos : {n_val}")
        print(f"  Test  pos : {n_test}")

    # ── Negative sampling ─────────────────────────────────────────────────
    # Use the FULL edge_index as exclusion set (train + val + test)
    # so no negative pair is actually an existing edge in any split
    def sample_neg(n_samples: int) -> torch.Tensor:
        """Sample n_samples non-edges, symmetric, excluding all known edges."""
        return negative_sampling(
            edge_index       = data.edge_index,  # exclude ALL known edges
            num_nodes        = num_nodes,
            num_neg_samples  = n_samples,
            force_undirected = True,             # keep pairs symmetric
        )

    train_neg = sample_neg(n_train)
    val_neg   = sample_neg(n_val)
    test_neg  = sample_neg(n_test)

    if verbose:
        print(f"\nNegative samples:")
        print(f"  Train neg : {train_neg.shape[1]}")
        print(f"  Val   neg : {val_neg.shape[1]}")
        print(f"  Test  neg : {test_neg.shape[1]}")

    # ── Package result ────────────────────────────────────────────────────
    result = {
        # Graph topology
        'train_edge_index' : train_edge_index,

        # Train split
        'train_pos_src'    : train_pos[0],
        'train_pos_dst'    : train_pos[1],
        'train_neg_src'    : train_neg[0],
        'train_neg_dst'    : train_neg[1],

        # Val split
        'val_pos_src'      : val_pos[0],
        'val_pos_dst'      : val_pos[1],
        'val_neg_src'      : val_neg[0],
        'val_neg_dst'      : val_neg[1],

        # Test split
        'test_pos_src'     : test_pos[0],
        'test_pos_dst'     : test_pos[1],
        'test_neg_src'     : test_neg[0],
        'test_neg_dst'     : test_neg[1],

        # Node info
        'node_ids'         : torch.arange(num_nodes, dtype=torch.long),
        'node_features'    : data.x.float(),
        'num_nodes'        : num_nodes,
        'feat_dim'         : feat_dim,
    }

    if verbose:
        print(f"\nNode features shape: {data.x.shape}")
        print(f"Data loading complete.\n")

    return result


# ─────────────────────────────────────────────────────────────
# Smoke test
# ─────────────────────────────────────────────────────────────

if __name__ == "__main__":
    d = load_and_split('Cora', verbose=True)

    print("Keys in result:")
    for k, v in d.items():
        if isinstance(v, torch.Tensor):
            print(f"  {k:25s}: shape {v.shape}, dtype {v.dtype}")
        else:
            print(f"  {k:25s}: {v}")

    # Verify no overlap between positive and negative edges
    def as_set(src, dst):
        return set(zip(src.tolist(), dst.tolist()))

    train_pos_set = as_set(d['train_pos_src'], d['train_pos_dst'])
    train_neg_set = as_set(d['train_neg_src'], d['train_neg_dst'])
    overlap       = train_pos_set & train_neg_set
    print(f"\nPos/neg overlap in train : {len(overlap)} (expected 0)")

Processing...


Dataset       : Cora
Nodes         : 2708
Edges         : 5278
Feature dim   : 1433

Edge split (80/10/10):
  Train pos : 4222
  Val   pos : 527
  Test  pos : 529

Negative samples:
  Train neg : 4222
  Val   neg : 526
  Test  neg : 528

Node features shape: torch.Size([2708, 1433])
Data loading complete.

Keys in result:
  train_edge_index         : shape torch.Size([2, 8444]), dtype torch.int64
  train_pos_src            : shape torch.Size([4222]), dtype torch.int64
  train_pos_dst            : shape torch.Size([4222]), dtype torch.int64
  train_neg_src            : shape torch.Size([4222]), dtype torch.int64
  train_neg_dst            : shape torch.Size([4222]), dtype torch.int64
  val_pos_src              : shape torch.Size([527]), dtype torch.int64
  val_pos_dst              : shape torch.Size([527]), dtype torch.int64
  val_neg_src              : shape torch.Size([526]), dtype torch.int64
  val_neg_dst              : shape torch.Size([526]), dtype torch.int64
  test_pos_src      

Done!


In [11]:
"""
Phase 1 — Structure-Based Learning
Step 5: Training Loop and Evaluation

Trains the ELSE encoder (Step 2) and GraphSAGE encoder (Step 3) jointly
using the edge-based binary cross-entropy loss L_S (Equation 5).

This is Stage 1 of the 3-stage SFDDGNN training procedure.
After this stage the weights of ELSE and GraphSAGE are frozen;
Stages 2 and 3 build on top of H_S produced here.

What is optimized:
    - ELSE encoder: DeepWalk, Node2Vec, SVD embedding tables + fusion MLP
    - GraphSAGE: conv1, conv2, edge_lin0, edge_lin1, readout

Stopping criterion:
    Early stopping with patience=50 epochs (paper Section 4.4):
    if val AP does not improve for 50 consecutive epochs, training stops
    and the best checkpoint is restored.

Usage:
    from step5_train_stage1 import train_stage1

    results = train_stage1(
        else_enc   = else_enc,
        struct_enc = struct_enc,
        data       = d,           # dict from step4_data.load_and_split
        device     = 'cuda',
    )
    # results['history']      — loss/AP/AUC per epoch
    # results['best_val_ap']  — best validation AP achieved
"""

import torch
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.metrics import average_precision_score, roc_auc_score




# ─────────────────────────────────────────────────────────────
# Evaluation
# ─────────────────────────────────────────────────────────────

@torch.no_grad()
def evaluate_stage1(
    else_enc:   ELSEEncoder,
    struct_enc: GraphSAGEStructureEncoder,
    node_ids:   torch.Tensor,
    edge_index: torch.Tensor,
    pos_src:    torch.Tensor,
    pos_dst:    torch.Tensor,
    neg_src:    torch.Tensor,
    neg_dst:    torch.Tensor,
    device:     str,
) -> dict:
    """
    Evaluate Stage 1 structure encoder on positive and negative edge pairs.

    During evaluation we use get_node_embeddings() (not forward_with_edges)
    so that val/test src/dst never pass through the edge update layers.
    Instead we score pairs by dot product on H_S — consistent with how
    H_S will be used in downstream stages.

    Args:
        else_enc   : ELSE encoder (produces X^B).
        struct_enc : GraphSAGE encoder (produces H_S from X^B).
        node_ids   : (N,) all node indices.
        edge_index : (2, E) training graph.
        pos_src/dst: positive edge endpoints.
        neg_src/dst: negative edge endpoints.
        device     : 'cuda' or 'cpu'.

    Returns:
        dict with 'ap' and 'auc' as percentage values.
    """
    else_enc.eval()
    struct_enc.eval()

    ni = node_ids.to(device)
    ei = edge_index.to(device)

    # Get full-graph node embeddings H_S
    x_b = else_enc(ni)                                  # (N, hidden_dim)
    H_S = struct_enc.get_node_embeddings(x_b, ei)       # (N, out_dim)

    # Score pairs via dot product on H_S
    src = torch.cat([pos_src, neg_src]).to(device)
    dst = torch.cat([pos_dst, neg_dst]).to(device)
    lbl = torch.cat([
        torch.ones(len(pos_src)),
        torch.zeros(len(neg_src)),
    ]).numpy()

    # Dot product similarity as link score
    scores = (H_S[src] * H_S[dst]).sum(dim=-1)
    probs  = torch.sigmoid(scores).cpu().numpy()

    return {
        'ap':  round(average_precision_score(lbl, probs) * 100, 2),
        'auc': round(roc_auc_score(lbl, probs) * 100, 2),
    }


# ─────────────────────────────────────────────────────────────
# Training loop
# ─────────────────────────────────────────────────────────────

def train_stage1(
    else_enc:    ELSEEncoder,
    struct_enc:  GraphSAGEStructureEncoder,
    data:        dict,
    device:      str   = 'cuda',
    lr:          float = 1e-3,
    weight_decay:float = 1e-5,
    batch_size:  int   = 1024,
    max_epochs:  int   = 500,
    patience:    int   = 50,
    checkpoint:  str   = '/tmp/stage1_best.pt',
    verbose:     bool  = True,
) -> dict:
    """
    Stage 1: Train ELSE + GraphSAGE with edge-based loss L_S (Eq. 5).

    Training procedure
    ------------------
    Each batch contains a mix of positive and negative edges:
        src = [pos_src_batch || neg_src_batch]
        dst = [pos_dst_batch || neg_dst_batch]
        lbl = [1, 1, ..., 0, 0, ...]

    Per batch:
        1. Recompute X^B = else_enc(all_node_ids)     — ELSE forward pass
        2. H_S, logits = struct_enc.forward_with_edges(X^B, train_ei, src, dst)
        3. loss = L_S(logits, labels)                 — Eq. 5
        4. loss.backward() + optimizer.step()

    Note: X^B is recomputed for ALL nodes every batch because both the
    ELSE embedding tables and MLP are being trained. This is necessary
    for correct gradients. The computational cost is manageable because
    ELSE is just embedding lookups + a small MLP.

    Early stopping:
        Track val AP. If it doesn't improve for `patience` epochs,
        restore the best checkpoint and stop.

    Args:
        else_enc     : ELSEEncoder from Step 2.
        struct_enc   : GraphSAGEStructureEncoder from Step 3.
        data         : dict from step4_data.load_and_split().
        device       : 'cuda' or 'cpu'.
        lr           : learning rate (paper: 1e-3).
        weight_decay : L2 regularisation (paper: 1e-5).
        batch_size   : edges per batch (paper: 1024).
        max_epochs   : maximum training epochs.
        patience     : early stopping patience (paper: 50).
        checkpoint   : path to save best model weights.
        verbose      : print epoch logs.

    Returns:
        dict:
            'history'     — list of dicts with loss/ap/auc per epoch
            'best_val_ap' — best validation AP achieved
            'best_epoch'  — epoch at which best AP occurred
    """
    # Move models to device
    else_enc   = else_enc.to(device)
    struct_enc = struct_enc.to(device)

    # Optimizer covers both ELSE and GraphSAGE parameters
    optimizer = optim.Adam(
        list(else_enc.parameters()) + list(struct_enc.parameters()),
        lr           = lr,
        weight_decay = weight_decay,
    )

    # Move fixed tensors to device
    node_ids   = data['node_ids'].to(device)
    train_ei   = data['train_edge_index'].to(device)

    # Build combined positive + negative training set
    all_src = torch.cat([data['train_pos_src'], data['train_neg_src']])
    all_dst = torch.cat([data['train_pos_dst'], data['train_neg_dst']])
    all_lbl = torch.cat([
        torch.ones(len(data['train_pos_src'])),
        torch.zeros(len(data['train_neg_src'])),
    ])

    loader = DataLoader(
        TensorDataset(all_src, all_dst, all_lbl),
        batch_size = batch_size,
        shuffle    = True,
    )

    # Training state
    history      = []
    best_val_ap  = 0.0
    best_epoch   = 0
    no_improve   = 0

    if verbose:
        n_params = (sum(p.numel() for p in else_enc.parameters()) +
                    sum(p.numel() for p in struct_enc.parameters()))
        print("=" * 60)
        print("Stage 1: Structure Pipeline Training")
        print("=" * 60)
        print(f"  Trainable params : {n_params:,}")
        print(f"  Batches / epoch  : {len(loader)}")
        print(f"  Max epochs       : {max_epochs}")
        print(f"  Patience         : {patience}")
        print(f"  Device           : {device}")
        print()

    for epoch in range(1, max_epochs + 1):
        # ── Training pass ────────────────────────────────────────────────
        else_enc.train()
        struct_enc.train()
        epoch_loss = 0.0

        for batch_src, batch_dst, batch_lbl in loader:
            batch_src = batch_src.to(device)
            batch_dst = batch_dst.to(device)
            batch_lbl = batch_lbl.to(device)

            optimizer.zero_grad()

            # Forward: ELSE → X^B for all nodes
            x_b = else_enc(node_ids)                           # (N, hidden_dim)

            # Forward: GraphSAGE → H_S + edge logits for this batch
            _H_S, logits = struct_enc.forward_with_edges(
                x_b, train_ei, batch_src, batch_dst
            )

            # Edge-based BCE loss (Eq. 5)
            loss = GraphSAGEStructureEncoder.structure_loss(logits, batch_lbl)

            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(loader)

        # ── Validation ───────────────────────────────────────────────────
        metrics = evaluate_stage1(
            else_enc, struct_enc,
            data['node_ids'], data['train_edge_index'],
            data['val_pos_src'], data['val_pos_dst'],
            data['val_neg_src'], data['val_neg_dst'],
            device,
        )

        history.append({
            'epoch'    : epoch,
            'loss'     : round(avg_loss, 6),
            'val_ap'   : metrics['ap'],
            'val_auc'  : metrics['auc'],
        })

        # ── Early stopping ────────────────────────────────────────────────
        if metrics['ap'] > best_val_ap:
            best_val_ap = metrics['ap']
            best_epoch  = epoch
            no_improve  = 0
            # Save best checkpoint (both models)
            torch.save({
                'else_enc'   : else_enc.state_dict(),
                'struct_enc' : struct_enc.state_dict(),
                'epoch'      : epoch,
                'val_ap'     : best_val_ap,
            }, checkpoint)
        else:
            no_improve += 1

        if verbose and epoch % 20 == 0:
            print(f"  Ep {epoch:4d}  "
                  f"loss={avg_loss:.4f}  "
                  f"val AP={metrics['ap']:.2f}%  "
                  f"val AUC={metrics['auc']:.2f}%  "
                  f"(best={best_val_ap:.2f}% @ ep {best_epoch})")

        if no_improve >= patience:
            if verbose:
                print(f"\n  Early stop at epoch {epoch}. "
                      f"No improvement for {patience} epochs.")
            break

    # Restore best checkpoint
    ckpt = torch.load(checkpoint)
    else_enc.load_state_dict(ckpt['else_enc'])
    struct_enc.load_state_dict(ckpt['struct_enc'])

    if verbose:
        print(f"\nStage 1 complete.")
        print(f"  Best val AP : {best_val_ap:.2f}%  (epoch {best_epoch})")

    return {
        'history'     : history,
        'best_val_ap' : best_val_ap,
        'best_epoch'  : best_epoch,
    }


# ─────────────────────────────────────────────────────────────
# Test evaluation helper
# ─────────────────────────────────────────────────────────────

@torch.no_grad()
def test_stage1(
    else_enc:   ELSEEncoder,
    struct_enc: GraphSAGEStructureEncoder,
    data:       dict,
    device:     str,
) -> dict:
    """
    Final test-set evaluation after Stage 1 training.

    Returns:
        dict with 'ap' and 'auc' as percentage values.
    """
    return evaluate_stage1(
        else_enc, struct_enc,
        data['node_ids'], data['train_edge_index'],
        data['test_pos_src'], data['test_pos_dst'],
        data['test_neg_src'], data['test_neg_dst'],
        device,
    )


# ─────────────────────────────────────────────────────────────
# Plotting helper
# ─────────────────────────────────────────────────────────────

def plot_stage1_history(history: list):
    """Plot training loss and val AP/AUC curves."""
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("matplotlib not installed — skipping plot")
        return

    epochs   = [h['epoch']   for h in history]
    losses   = [h['loss']    for h in history]
    val_aps  = [h['val_ap']  for h in history]
    val_aucs = [h['val_auc'] for h in history]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, losses, color='steelblue', linewidth=1.5)
    axes[0].set_title('Stage 1 — Training Loss (L_S)')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('BCE Loss')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, val_aps,  label='AP',  color='steelblue', linewidth=1.5)
    axes[1].plot(epochs, val_aucs, label='AUC', color='coral',     linewidth=1.5)
    axes[1].set_title('Stage 1 — Validation Metrics')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('% Score')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('stage1_training.png', dpi=150)
    print("Plot saved to stage1_training.png")
    plt.show()

In [ ]:
"""
Phase 1 — Structure-Based Learning
Main Runner (Colab-compatible)

No argparse — edit the CONFIG dict below then run the cell.

Steps executed:
    1 + 2  →  DeepWalk, Node2Vec, SVD-MF  →  ELSE encoder
    3      →  GraphSAGE structure encoder
    4      →  Load dataset, 80/10/10 edge split
    5      →  Train Stage 1, evaluate on test set
"""

import sys
import torch

# ── Make sure step files are importable ──────────────────────────────────
# Adjust PHASE1_DIR to wherever you uploaded the step*.py files.
PHASE1_DIR = '/content/phase1'
if PHASE1_DIR not in sys.path:
    sys.path.insert(0, PHASE1_DIR)




# ════════════════════════════════════════════════════════════════
# CONFIG — edit values here, then re-run the cell
# ════════════════════════════════════════════════════════════════
CONFIG = {
    # ── Dataset ───────────────────────────────────────────────
    'dataset'     : 'Cora',       # 'Cora' | 'PubMed' | 'CiteSeer'
    'data_root'   : './data',
    'seed'        : 42,

    # ── Graph embedding algorithms ────────────────────────────
    'basis_dim'   : 64,    # embedding dim for each of DeepWalk / Node2Vec / SVD
    'num_walks'   : 10,    # random walks per node
    'walk_length' : 80,    # steps per walk
    'n2v_p'       : 1.0,   # Node2Vec return parameter
    'n2v_q'       : 0.5,   # Node2Vec in-out parameter (0.5 = slightly DFS-biased)
    'use_ppmi'    : True,  # PPMI normalisation for SVD

    # ── Model architecture ────────────────────────────────────
    'hidden_dim'  : 256,   # ELSE output dim = GraphSAGE hidden dim
    'struct_dim'  : 128,   # GraphSAGE output dim  (H_S dimension)
    'dropout'     : 0.5,

    # ── Training ──────────────────────────────────────────────
    'device'      : 'cuda' if torch.cuda.is_available() else 'cpu',
    'lr'          : 1e-3,
    'weight_decay': 1e-5,
    'batch_size'  : 1024,
    'max_epochs'  : 500,
    'patience'    : 50,
    'checkpoint'  : '/tmp/stage1_best.pt',

    # ── Output ────────────────────────────────────────────────
    'plot'        : True,
}
# ════════════════════════════════════════════════════════════════


def run():
    c = CONFIG
    print(f"Device : {c['device']}")

    # ── Step 4: Load data ────────────────────────────────────────────────
    print("\n" + "─" * 55)
    print("Step 4: Loading dataset")
    print("─" * 55)
    data = load_and_split(
        dataset_name = c['dataset'],
        data_root    = c['data_root'],
        seed         = c['seed'],
        verbose      = True,
    )

    # ── Steps 1 & 2: ELSE encoder ────────────────────────────────────────
    print("\n" + "─" * 55)
    print("Steps 1 & 2: Building ELSE encoder")
    print("─" * 55)
    else_enc = build_else_encoder(
        edge_index   = data['train_edge_index'],
        num_nodes    = data['num_nodes'],
        basis_dim    = c['basis_dim'],
        output_dim   = c['hidden_dim'],
        num_walks    = c['num_walks'],
        walk_length  = c['walk_length'],
        n2v_p        = c['n2v_p'],
        n2v_q        = c['n2v_q'],
        use_ppmi     = c['use_ppmi'],
        trainable    = True,
        verbose      = True,
    )

    # ── Step 3: GraphSAGE encoder ────────────────────────────────────────
    print("\n" + "─" * 55)
    print("Step 3: Building GraphSAGE structure encoder")
    print("─" * 55)
    struct_enc = GraphSAGEStructureEncoder(
        in_dim     = c['hidden_dim'],
        hidden_dim = c['hidden_dim'],
        out_dim    = c['struct_dim'],
        dropout    = c['dropout'],
    )
    n_params = sum(p.numel() for p in struct_enc.parameters())
    print(f"GraphSAGE params : {n_params:,}")

    # ── Step 5: Train ────────────────────────────────────────────────────
    print("\n" + "─" * 55)
    print("Step 5: Training Stage 1 — Structure Pipeline")
    print("─" * 55)
    results = train_stage1(
        else_enc     = else_enc,
        struct_enc   = struct_enc,
        data         = data,
        device       = c['device'],
        lr           = c['lr'],
        weight_decay = c['weight_decay'],
        batch_size   = c['batch_size'],
        max_epochs   = c['max_epochs'],
        patience     = c['patience'],
        checkpoint   = c['checkpoint'],
        verbose      = True,
    )

    # ── Test evaluation ───────────────────────────────────────────────────
    print("\n" + "─" * 55)
    print("Test set evaluation")
    print("─" * 55)
    test_m = test_stage1(else_enc, struct_enc, data, c['device'])
    print(f"  Dataset  : {c['dataset']}")
    print(f"  Test AP  : {test_m['ap']:.2f}%")
    print(f"  Test AUC : {test_m['auc']:.2f}%")

    # ── Plot ──────────────────────────────────────────────────────────────
    if c['plot']:
        plot_stage1_history(results['history'])

    return else_enc, struct_enc, data, results


# Run immediately when this cell is executed
else_enc, struct_enc, data, results = run()

Device : cuda

───────────────────────────────────────────────────────
Step 4: Loading dataset
───────────────────────────────────────────────────────
Dataset       : Cora
Nodes         : 2708
Edges         : 5278
Feature dim   : 1433

Edge split (80/10/10):
  Train pos : 4222
  Val   pos : 527
  Test  pos : 529

Negative samples:
  Train neg : 4222
  Val   neg : 526
  Test  neg : 528

Node features shape: torch.Size([2708, 1433])
Data loading complete.


───────────────────────────────────────────────────────
Steps 1 & 2: Building ELSE encoder
───────────────────────────────────────────────────────
Building ELSE Encoder
[DeepWalk] Building adjacency list...
[DeepWalk] Generating 27,080 walks (10 per node, length 80)...
[DeepWalk] Training Skip-gram on 27,080 walks (window=5, negatives=5)...
